# CONCLAVE — Phase 2: Consensus, Projection & Flagging

This notebook runs **Phase 2**: takes your annotated clusters from Phase 1 and turns them into
a single labeled cell-type call for *every* cell in your full dataset, with quality metrics.

**What Phase 2 actually does, step by step:**

1. Loads your filled-in `annotation_template_<method>.csv` files (cluster ID → your cell-type label)
2. Applies those labels to Phase 1's clustered subsample, per method
3. Computes **consensus**: majority vote across your methods (needs `MIN_VOTES` methods agreeing)
4. Builds a balanced template (up to `TEMPLATE_MAX_PER_LABEL` cells per consensus cell-type)
5. Fits a 3D UMAP + KNN model on that template
6. **Projects consensus labels onto your full dataset** (not just the clustered subsample) — every
   cell gets a predicted cell type + confidence score
7. Repeats the projection separately for each individual method, so you can compare methods
   against consensus
8. Computes quality metrics: disagreement score, confidence scores, Jensen-Shannon Divergence
   (consensus vs. each method, overall and per-sample)
9. Generates ~16 diagnostic plots
10. Saves everything, most importantly `full_dataset_labeled_complete.csv`

> **Known limitation:** `run_phase2_complete()` doesn't yet accept your data as normal function
> arguments — it reads configuration from module-level variables that must be set before calling
> it. This notebook handles that for you; see `README.md` for the full explanation if you're
> curious. This is on the list for a proper refactor.

> **Runtime:** Phase 2 fits models on your balanced template, then projects onto your **full**
> dataset (not the Phase 1 subsample) once per method plus once for consensus. On ~97,000 cells
> with 3 methods, this took about 5 minutes in testing. Runtime scales with your full dataset
> size and number of methods.

## Step 0 — Setup

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

import conclave.phase2.pipeline_complete as p2

## Step 1 — Point to your Phase 1 outputs and annotations

- `PHASE1_OUTPUT`: the `outdir` you used in Phase 1
- `ANNOTATIONS_DIR`: a folder containing your filled-in annotation CSVs, one per method, each
  with `cluster_id`, `n_cells`, `annotation` columns (copy/rename from
  `PHASE1_OUTPUT/04_cluster_heatmaps/annotation_template_<method>.csv` once you've filled in the
  `annotation` column by hand)

In [ ]:
# ⚙️ config
PHASE1_OUTPUT = Path("./phase1_output")
PHASE2_OUTPUT = Path("./phase2_output")
ANNOTATIONS_DIR = Path("./annotations")

# Which methods to include -- must match method names you have BOTH a
# label_<method> column for (in Phase 1's clustered subset) AND a filled-in
# annotation file for
CONSENSUS_METHODS = ["phenograph", "kmeans", "minibatchkmeans"]

ANNOTATION_FILES = {
    method: ANNOTATIONS_DIR / f"{method}_annotated.csv"
    for method in CONSENSUS_METHODS
}

CLUSTERED_FILE = PHASE1_OUTPUT / "03_clustering_annotation" / "clustered_subset_with_labels_on_sampled.csv"
FULL_DATA_FILE = PHASE1_OUTPUT / "01_normalized_full.csv"

MARKERS = [
    'CD34', 'CD31', 'CD141', 'PNAd', 'CD25', 'CD14', 'CD1c', 'CK', 'CD21',
    'FoxP3', 'CD23', 'GRB7', 'CD1A', 'Podoplanin', 'CD138', 'CD248', 'CD64', 'CD163',
    'Pax5', 'IRF8', 'CD20', 'CD8', 'CD303', 'LYZ', 'CD16', 'CD2', 'HLADR', 'IRF4', 'CD5',
    'CD79a', 'CD68', 'CD3', 'CD4', 'CD27', 'PRDM1', 'MELANA', 'S100B',
]  # should match what you used in Phase 1

SAMPLE_COLS = ["ID"]  # should match what you used in Phase 1

KNN_K = 25
MIN_VOTES = 2                 # how many methods must agree for a cell to get a consensus label
TEMPLATE_MAX_PER_LABEL = 500  # cap on cells per cell-type in the balanced template

## Step 2 — Validate before running

Catches the most common mistakes (missing files, empty annotations, mismatched cluster IDs)
*before* committing to a multi-minute run.

In [ ]:
problems = []

if not CLUSTERED_FILE.exists():
    problems.append(f"Missing clustered file: {CLUSTERED_FILE}")
if not FULL_DATA_FILE.exists():
    problems.append(f"Missing full data file: {FULL_DATA_FILE}")

if not problems:
    clustered = pd.read_csv(CLUSTERED_FILE, nrows=5)

for method in CONSENSUS_METHODS:
    ann_path = ANNOTATION_FILES[method]
    if not ann_path.exists():
        problems.append(f"Missing annotation file for '{method}': {ann_path}")
        continue

    ann = pd.read_csv(ann_path)
    if "annotation" not in ann.columns:
        problems.append(f"{ann_path.name}: no 'annotation' column")
        continue

    n_blank = ann["annotation"].isna().sum() + (ann["annotation"].astype(str).str.strip() == "").sum()
    if n_blank > 0:
        problems.append(f"{ann_path.name}: {n_blank}/{len(ann)} rows have a blank 'annotation' -- fill these in")

    label_col = f"label_{method}"
    if not problems and label_col not in clustered.columns:
        problems.append(f"Phase 1 clustered file has no '{label_col}' column -- was '{method}' actually run in Phase 1?")
    elif not problems:
        clustered_full = pd.read_csv(CLUSTERED_FILE, usecols=[label_col])
        clustered_ids = set(clustered_full[label_col].unique())
        annotated_ids = set(ann["cluster_id"].unique())
        missing_annotation = clustered_ids - annotated_ids
        extra_annotation = annotated_ids - clustered_ids
        if missing_annotation:
            problems.append(f"{method}: cluster(s) {sorted(missing_annotation)} exist in Phase 1 output but have no row in {ann_path.name}")
        if extra_annotation:
            problems.append(f"{method}: {ann_path.name} has cluster_id(s) {sorted(extra_annotation)} that don't exist in Phase 1 output (typo?)")

if problems:
    print("⚠️  Found issues -- fix these before running Step 3:")
    for p in problems:
        print(f"  - {p}")
else:
    print(f"✅ All checks passed for {len(CONSENSUS_METHODS)} methods: {CONSENSUS_METHODS}")

## Step 3 — Run Phase 2

This is the multi-minute step -- see the runtime note in the intro.

In [ ]:
assert not problems, "Fix the issues listed in Step 2 before running this cell."

p2.PHASE1_OUTPUT = PHASE1_OUTPUT
p2.PHASE2_OUTPUT = PHASE2_OUTPUT
p2.ANNOTATIONS_DIR = ANNOTATIONS_DIR
p2.CLUSTERED_FILE = CLUSTERED_FILE
p2.FULL_DATA_FILE = FULL_DATA_FILE
p2.ANNOTATION_FILES = ANNOTATION_FILES
p2.MARKERS = MARKERS
p2.CONSENSUS_METHODS = CONSENSUS_METHODS
p2.KNN_K = KNN_K
p2.MIN_VOTES = MIN_VOTES
p2.SAMPLE_COLS = SAMPLE_COLS
p2.TEMPLATE_MAX_PER_LABEL = TEMPLATE_MAX_PER_LABEL

# Output dirs are created at import time using the OLD default path --
# recreate them at your actual PHASE2_OUTPUT location
p2.PHASE2_OUTPUT.mkdir(parents=True, exist_ok=True)
(p2.PHASE2_OUTPUT / "templates").mkdir(exist_ok=True)
(p2.PHASE2_OUTPUT / "plots").mkdir(exist_ok=True)

df_labeled, template, single_templates, report = p2.run_phase2_complete()

## Step 4 — Review results

In [ ]:
print(f"Cells labeled: {len(df_labeled):,}")
print(f"Consensus mean confidence: {df_labeled['confidence_score'].mean():.3f}")
print(f"High confidence (>0.8): {(df_labeled['confidence_score'] > 0.8).mean()*100:.1f}%")
print(f"Full disagreement (all methods differ): {df_labeled['absolute_no_consensus'].mean()*100:.1f}%")
print()
print("Consensus label distribution:")
print(df_labeled["consensus_label"].value_counts())

In [ ]:
plots_dir = PHASE2_OUTPUT / "plots"
for name in ["confidence_distribution.png", "disagreement_ranked_RED.png", "umap_3d_consensus.png"]:
    p = plots_dir / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))

## Reference: all Phase 2 outputs

```
phase2_output/
├── full_dataset_labeled_complete.csv  ← every cell, every method's projected label + consensus + confidence + flags
├── template_with_flags.csv            ← the balanced consensus template, with QC flags
├── consensus_template.csv
├── metrics_summary.csv
├── phase2_complete_report.json
├── umap_model_consensus.pkl
├── templates/
│   ├── template_<method>.csv          (one per method)
│   └── umap_model_<method>.pkl
└── plots/                             (~16 diagnostic plots)
    ├── confidence_distribution.png
    ├── confidence_all_methods.png
    ├── disagreement_ranked_RED.png
    ├── disagreement_by_sample.png / _flagged.png
    ├── disagreement_full_dataset_RED.png
    ├── jsd_comparison.png / jsd_pairwise_heatmap.png / jsd_heatmap_samples_celltypes.png
    ├── jsd_mean_per_sample_scatter.png
    ├── spatial_confidence_heatmap_tiles.png / _per_sample.png
    └── umap_3d_<consensus|method>.png
```

**Key columns in `full_dataset_labeled_complete.csv`:**
- `consensus_label`, `confidence_score` — the main result
- `<method>_label_projected`, `<method>_confidence` — per-method projections, for comparison
- `disagreement_score_full` — 0 (all methods agree), 1 (partial), 2 (all differ)
- `absolute_no_consensus` — 1 if disagreement_score_full == 2 (equivalently: no consensus label)
